In [5]:
import requests

response = requests.post(
    "http://localhost:11434/api/show",
    json={
        "model":"qwen3.5:9b",
        "verbose":True
    }
)

data = response.json()

print("모델 크기: ",data['details']['parameter_size'])
print("양자화: ",data['details']['quantization_level'])
print("모델 형식: ",data['details']['format'])

모델 크기:  9.0B
양자화:  Q4_K_M
모델 형식:  gguf


In [6]:
OLLAMA_HOST="http://localhost:11434"
MODEL = "qwen3.5:9b"

#### 방법 1. requests 라이브러리 사용

In [7]:
response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model":MODEL,
        "messages":[
            {
                "role":"user",
                "content":"안녕하세요"
            }
        ],
        "stream":False,
        "think":False # 추론 하지 마
    }
)

result = response.json()
print(result['message']['content'])

안녕하세요! 반갑습니다. 👋
오늘 어떤 도움이 필요하신가요? 궁금한 점이 있거나, 대화하고 싶은 주제가 있다면 언제든지 말씀해 주세요.


#### 방법 2. Ollama python 라이브러리 사용

In [8]:
from ollama import Client

client = Client(host=OLLAMA_HOST)

In [9]:
response = client.chat(
    model=MODEL,
    messages=[
            {
                "role":"user",
                "content":"안녕하세요!! 간단히 자기 소개 해줘"
            }
    ],
    stream=False,
    think=False # 추론 하지 마
)

response.message.content

'안녕하세요! 👋 저는 **Qwen3.5**, 알리바바 클라우드가 개발한 초대규모 언어 모델입니다. 이전 버전보다 더 지식을 넓히고, 논리적 추론과 시각 분석 능률을 높였으며, 코딩과 장문 컨텍스트 이해도 크게 개선되었습니다.\n\n저와 함께 복잡한 문제를 해결하거나 창작 활동을 하시는 건 어떠신가요? 😊'

In [10]:
# temperature=0 : 확률이 높은 토큰을 선택
# temperature=1.5 : 다양하게 낮은 토큰도 선택할 수 있음(실험적인 텍스트 생성)

def ask(messages,think=False, temperature=0):
    response = client.chat(
    model=MODEL,
    messages=messages,
    stream=False,
    think=False, 
    options={"temperature":temperature}
)
    return response.message.content

In [8]:
messages=[
            {
                "role":"user",
                "content":"LLM을 한줄요약해줘"
            }
    ]
ask(messages=messages)

'LLM(대규모 언어 모델)은 방대한 양의 텍스트 데이터를 학습하여 인간의 언어를 이해하고 생성할 수 있는 인공지능 모델입니다.'

In [9]:
messages=[
            {
                "role":"user",
                "content":"파이썬을 설명해줘"
            }
    ]

response = client.chat(
    model=MODEL,
    messages=messages,
    stream=True,
    think=False, 
)

for chunk in response:
    print(chunk['message']['content'],end="",flush=True)

**파이썾 (Python)** 은 전 세계적으로 가장 인기 있고 널리 쓰이는 **고수준 인터프리팅 객체 지향 프로그래밍 언어**입니다. 1980  년에서 2000 년 사이에 처음 개발되었으며, 이후 마이크로소프트와 구글 같은 거대 기업에서부터 개인 개발자까지 폭넓게 사용되고 있습니다.

파이썾을 한 마디로 설명하자면 **"사용자가 읽기 쉽도록 설계된, 거의 영어에 가까운 자연스러운 문법"** 을 가진 언어라고 할 수 있습니다.

주요 특징과 장점을 자세히 살펴보면 다음과 같습니다:

### 1. 직관적이고 쉬운 문법
- 다른 언어 (C++, 자바 등) 에 비해 불필요한 괄호나 세미콜론 (;) 등의 구문 제한이 적습니다.
- 코드가 짧고 간결하게 작성할 수 있어 초보자도 비교적 빨리 핵심 로직을 이해하고 만들 수 있습니다.

### 2. 강력한 생태계와 라이브러리
- "바퀴를 다시 발명하지 마라 (Don't reinvent the wheel)"는 파이썾의 철학대로, 이미 완성된 기능을 제공하는 방대한 **라이브러리**가 많습니다.
- 과학 계산 (`numpy`, `pandas`), 웹 개발 (`Django`, `Flask`), 인공지능 및 머신러닝 (`TensorFlow`, `PyTorch`) 등 거의 모든 분야에서 활용됩니다.

### 3. 플랫폼 독립성 (크로스 플랫폼)
- OS 와 무관하게 실행될 수 있습니다. 윈도우, 맥OS, 리눅스 등에서 동일한 파이썾 코드를 그대로 실행할 수 있어 개발 환경 설정이 매우 간편합니다.

### 4. 멀티플래닝 기능 지원
- 하나의 프로그램 내에서 여러 개의 코드 블록 (프로세스) 을 동시에 처리할 수 있어 복잡한 연산이나 백그라운드 작업을 효율적으로 할 때 유리합니다.

---

### 간단한 예제 코드
파이썾의 문법이 얼마나 간단해 보이는지 예를 들어보겠습니다. 위아래에서 바로 실행 가능한 코드입니다.

```python
# 이름 입력 받기
이름 = input("당신의 이름은 무엇인가요? ")

# 인사말 출력 (엔터

In [10]:
# 프롬프트 작성 시 기본개념 1) role 지정 - system
messages=[
{"role":"system","content":"초등학생에게 설명하듯이 쉽게 설명해줘."},
{"role":"user","content":"인공지능을 설명해줘"},
]

ask(messages=messages)

KeyboardInterrupt: 

In [ ]:
# 프롬프트 작성 시 예시 여부에 따른 zero shot, few shot

In [1]:
messages=[
{"role":"system","content":"너는 문장의 감성을 긍정,부정,중립 중 하나로 분류하는 AI야"},
{"role":"user","content":"정말 만족스러운 서비스였어요."},
{"role":"assistant","content":"긍정"},
{"role":"user","content":"다시는 이용 안할 것 같아요."},
{"role":"assistant","content":"부정"},
{"role":"user","content":"배송이 너무 늦었어요."},
]

ask(messages)


NameError: name 'ask' is not defined

In [ ]:
messages=[
    {"role":"user","content":"대한민국 2026 년 대통령"}
]

ask(messages)

'현재 시점에서 **2026 년 대한민국 대통령을 특정할 수는 없습니다.**\n\n대통령 선거는 정해진 일정 (2027 년 3 월) 에 국민투표를 통해 결정되기 때문입니다. 다만, 다음과 같은 중요한 사실들을 정리해 드릴 수 있습니다:\n\n1.  **임기 시작일**: 대한민국 제 20 대 대통령 선거는 **2027 년 3 월 9 일**에 치러지며, 당선자는 그 해 5 월 10 일에 취임합니다. 따라서 2026 년에는 아직 새로운 대통령이 선출되지 않은 상태입니다.\n2.  **현직 대통령**: 현재 (2024 년 기준) 대한민국 대통령은 윤석열 대통령이며, 그의 임기는 2027 년 5 월까지 이어집니다.\n3.  **예측 불가**: 선거 결과는 유권자의 투표에 의해 결정되므로, 누구도 정확한 결과를 미리 알 수 없습니다.\n\n혹시 특정 후보자에 대한 정보나 선거 제도 관련 궁금증이 있으신가요?'

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df  = pd.read_csv("../data/reviews.csv")

# 학습 데이터 와 테스트 데이터 분리 
# 데이터 100개 중 80개를 시험공부 , 20개를 모의고사 문제로 나누기 (데이터 8:2로 나누는 예시)
text_train, text_test, y_train, y_test = train_test_split(df['text'],df['label'],test_size=0.2,random_state=42,stratify=df['label'])

SYSTEM="당신은 영화리뷰 감정 분석가입니다. 리뷰를 읽고, '긍정' 또는 '부정' 한 단어로만 답해주세요."

def classify(review,shots=None):
    messages = [{"role":"system","content":SYSTEM}]

    for text, label in (shots or []):
        messages.append({"role":"user", "content":f"리뷰 {text}"})
        messages.append({"role":"assistant", "content":f"{label}"})
    messages.append({"role":"user", "content":f"리뷰 {review}"})

    answer = ask(messages)

    if "긍정" in answer: return 1
    if "부정" in answer: return 0

    return -1


In [ ]:
sample, y_sample = text_test[:100], y_test[:100]
preds = [ classify(t) for t in sample]
acc =  (pd.Series(preds, index=y_sample.index) == y_sample).mean()
print("제로샷 정확도:",round(acc, 3), "/ 형식 어긴 답 :",preds.count(-1))

제로샷 정확도: 0.92 / 형식 어긴 답 : 0


In [ ]:
shots = [
    ("배우들 연기가 살아있다. 강추","긍정"),
    ("시간아깝다 진짜","부정"),
    ("ost만 기억에 남는다","부정"),
    ("두 번 봐도 또 울었다","긍정"),
]

preds_few = [classify(t,shots) for t in sample]
acc =  (pd.Series(preds_few, index=y_sample.index) == y_sample).mean()
print("퓨 샷 정확도:",round(acc, 3), "/ 형식 어긴 답 :",preds_few.count(-1))

퓨 샷 정확도: 0.89 / 형식 어긴 답 : 0


In [ ]:
messages=[
    {"role":"user","content":"사내 AI 스터디 이름을 하나 작성해"}
]

for t in [0,.1,1.5]:
    print(t, [ask(messages=messages,temperature=t)for _ in range(3)])

0 ['사내 AI 스터디의 분위기와 목적에 맞춰 선택하실 수 있도록 몇 가지 컨셉별로 이름안을 제안해 드립니다.\n\n### 1. 직관적이고 명확한 스타일 (신뢰감 강조)\n*   **AI 인사이트 (AI Insight)**: 단순 학습을 넘어 통찰력을 얻는다는 의미\n*   **스마트 브레인 (Smart Brain)**: 조직의 지능을 높이는 곳이라는 뜻\n*   **AI 아카데미 (AI Academy)**: 체계적인 교육과 성장을 강조\n*   **데이터 파이터 (Data Fighter)**: 데이터를 통해 문제를 해결하는 팀이라는 느낌\n\n### 2. 친근하고 유쾌한 스타일 (팀워크 강조)\n*   **알리바바 (Alibaba)**: \'알\'이 AI, \'바\'가 배움을 뜻하는 약자 (또는 알리바바 그룹의 성공을 빌기)\n*   **AI 나우 (AI Now)**: 지금 바로 시작한다는 의미\n*   **스마트톡 (Smart Talk)**: AI 관련 대화와 토론을 강조\n*   **코드 & 커피 (Code & Coffee)**: 기술과 휴식을 함께한다는 느낌\n\n### 3. 미래 지향적이고 모티베이션 있는 스타일\n*   **뉴노멀 AI (New Normal AI)**: 변화하는 시대에 맞는 새로운 접근법\n*   **인사이트 X (Insight X)**: 무한한 가능성과 확장성 강조\n*   **퓨처 머신러닝 (Future ML)**: 미래 기술을 선도한다는 의미\n*   **AI 스프린트 (AI Sprint)**: 빠르게 성장하고 발전한다는 뜻\n\n### 4. 영어/혼용 이름 (글로벌 이미지)\n*   **Deep Dive AI**: 깊이 있는 학습을 강조\n*   **Neural Connect**: 신경망처럼 서로 연결되어 지식을 공유함\n*   **Synapse Lab**: 시냅스처럼 아이디어가 연결되는 실험실\n\n**💡 팁:** 팀원들의 반응이 좋을 만한 이름으로 정하기 위해, 최종 결정 전에 "이름 투표"를 하거나 슬로건 (예: "AI

In [ ]:
import time

messages=[
    {"role":"user","content":"사과 3개에 배 2개를 더하고 두 개를 먹으면 과일은 몇 개?"}
]

for think in [False, True]:
    start = time.time()
    r = client.chat(
        model=MODEL,
        messages=messages,
        stream=False,
        think=think
)
    print("think=",think,"시간 : ",round(time.time() - start),"초")

    if think:
        print("생각과정 : ", (r.message.thinking or "")[:200])
    print("답:",r.message.content)

think= False 시간 :  3 초
답: 먼저 주어진 과일의 총 수량을 구해봅시다.
사과 3 개와 배 2 개를 더하면, 처음에는 **5 개**의 과일이 있게 됩니다 (3 + 2 = 5).

그다음 이 과일 중 **2 개**를 먹으므로, 남은 과일의 수는 5 개에서 2 개를 뺍니다.
5 - 2 = 3

따라서 남는 과일은 **3 개**입니다.
think= True 시간 :  32 초
생각과정 :  Thinking Process:

1.  **Analyze the Request:**
    *   Input: "사과 3 개에 배 2 개를 더하고 두 개를 먹으면 과일은 몇 개?" (Apples 3 plus Pears 2, then eat two, how many fruits are left?)
    *   Task: Solve the arithmeti
답: 사과 3 개에 배 2 개를 더하면 과일은 총 **5 개**가 됩니다.

그 중 두 개를 먹으면:
**5 - 2 = 3**

따라서 남은 과일은 **3 개**입니다.


#### 출력형식 json, pydantic

In [ ]:
import json

schema = {
    "type":"object",
    "properties":{
        "label":{"type":"string","enum":['긍정','부정']},
        "reason":{"type":"string"},
    },
    "required":["label","reason"]
}

# generate() :
r = client.generate(
        model=MODEL,
        prompt=f"{SYSTEM}\n이유도 한 문장으로 적어줘\n리뷰:연기가 좋고 정말 재밌어요.",
        stream=False,
        think=False,
        format=schema,
        options={"temperature":0}
)
print(r.response)
result = json.loads(r.response)

{
  "label": "긍정",
  "reason": "연기와 재미를 칭찬하는 긍정적인 표현이 포함되어 있습니다."
}


In [ ]:
print(result['label'],"|",result['reason'])

긍정 | 연기와 재미를 칭찬하는 긍정적인 표현이 포함되어 있습니다.


In [ ]:
from pydantic import BaseModel, Field
# schema = {
#     "type":"object",
#     "properties":{
#         "label":{"type":"string","enum":['긍정','부정']},
#         "reason":{"type":"string"},
#     },
#     "required":["label","reason"]
# }

class SentimentResult(BaseModel):
    sentiment:str
    score:float = Field(ge=0.0, le=1.0)
prompt = """
다음 문장의 감정을 분석하라.

문장:
오늘 정말 행복한 하루였다.
"""

result = client.chat(model=MODEL, messages=[{"role":"user","content":prompt}], format=SentimentResult.model_json_schema())
print(result)

# 답변 형식 점검
response = SentimentResult.model_validate_json(result.message.content)
print(response.sentiment)
print(response.score)

model='qwen3.5:9b' created_at='2026-11-06T07:07:49.831735396Z' done=True done_reason='stop' total_duration=19918376708 load_duration=3873375 prompt_eval_count=29 prompt_eval_duration=86390000 eval_count=696 eval_duration=19708885000 message=Message(role='assistant', content='{\n  "sentiment": "긍정적",\n  "score": 0.95\n}', thinking='Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Input: A Korean sentence ("오늘 정말 행복한 하루였다.").\n    *   Task: Analyze the emotion of the sentence.\n    *   Output Language: Korean (implied by the input and request).\n\n2.  **Analyze the Sentence:**\n    *   Sentence: "오늘 정말 행복한 하루였다." (Today was really a happy day.)\n    *   Key Word: "행복한" (happy/happiness).\n    *   Tone: Positive, affirmative ("오늘", "정말").\n    *   Emotion: Joy, happiness, satisfaction, contentment.\n\n3.  **Determine the Emotion Category:**\n    *   Primary emotion: Positive (긍정적).\n    *   Specific emotion: Happiness/Joy (행복/기쁨).\n\n4.  **Draft the Analysis:**\n    *   Identify

In [ ]:
# 뉴스 분석 : summary(str), sentiment(positive,negative, neutral), keyword[과학,기술,]
from typing import Literal

class NewsResult(BaseModel):
    summary:str
    sentiment:Literal['positive','negative','neutral']
    keyword:list[str]


In [ ]:
prompt = """
다음 뉴스 기사를 분석해줘.
단, format 으로 주어진 구조로 출력해줘.

기사 : 
법원 "'이용자 1천만 명 기록 무단수집' 쿠팡 과징금 2천억 효력정지"
1천만 명에 이르는 이용자들의 온라인 기록을 무단으로 수집해 사용했다는 이유로 쿠팡에 부과된 2천억 원대 과징금 처분의 효력이 정지됐습니다.
서울행정법원 행정1부는 오늘 쿠팡이 개인정보보호위원회를 상대로 "'과징금 부과 및 시정·공표명령 처분 효력을 정지해 달라"며 낸 집행정지 신청을 인용했습니다.
이 처분의 효력은 과징금 취소 청구 본안 소송의 1심 판결 선고일로부터 30일이 될 때까지 정지됩니다.
재판부는 "개보위의 처분으로 쿠팡에 회복하기 어려운 손해가 발생할 우려가 있다"며 "효력정지로 공공복리에 중대한 영향을 미칠 우려가 있는 경우로도 보이지 않는다"고 판단했습니다.
앞서 개인정보보호위원회는 쿠팡이 온라인 이용자 1천117만 명의 온라인 활동기록을 법적 근거 없이 수집·이용한 사실을 적발해 과징금 2천11억 원을 부과한 바 있습니다.
한편 어제 행정법원의 다른 재판부는 3750만 명의 개인정보를 유출한 쿠팡에 개보위가 부과한 4천346억 원 규모의 과징금에 대해선 효력을 유지하라고 했습니다.
"""

result = client.chat(model=MODEL, messages=[{"role":"user","content":prompt}], format=NewsResult.model_json_schema())
print(result)

# 답변 형식 점검
response = NewsResult.model_validate_json(result.message.content)
print(response.sentiment)
print(response.keyword)

model='qwen3.5:9b' created_at='2026-11-06T07:24:45.971505567Z' done=True done_reason='stop' total_duration=76933543173 load_duration=2817228 prompt_eval_count=344 prompt_eval_duration=87259000 eval_count=2757 eval_duration=76671069000 message=Message(role='assistant', content='{\n  "summary": "",\n  "sentiment": "neutral",\n  "keyword": [\n    "쿠팡",\n    "과징금",\n    "효력정지",\n    "개인정보보호위원회",\n    "서울행정법원"\n  ]\n}', thinking='Okay, the user wants me to analyze a news article and output it in a specific format they provided. But looking at the current conversation, there\'s no actual format example given. Wait, the user mentioned "format 으로 주어진 구조로 출력해줘" which means "output in the structure given for the format". However, in the initial problem description, there\'s no explicit format example. Maybe I missed something?\n\nWait, looking back at the query: "단, format 으로 주어진 구조로 출력해줘." So the user expects me to know the format that was previously provided. But since this is a new conversati

In [11]:
history = [{"role":"system","content":"당신은 친절한 사내 도우미이다. 한국어로 3문장 이내로 답한다."}]

while True:
    q = input("나: ")
    if q in ("q","종료"):
        break

    history.append({"role":"user","content":q})
    print("AI: ", end="")
    answer = ""

    for chunk in client.chat(model=MODEL, messages=history, stream=True, think=False):
        content = chunk.message.content
        print(content,end="",flush=True)
        answer += content

    print()
    #history.append({"role":"assistant","content":answer})

AI: 안녕하세요, 도와드릴 일이 있으신가요?  
궁금한 점이 있거나 도움이 필요하시면 언제든 말씀해 주세요.  
오늘 하루도 편안하게 보내시길 응원하겠습니다.
AI: 당신은 "1234"라는 숫자만 말씀하셨네요. 

이 번호가 특별한 의미나 필요한 정보가 있는지 궁금하시다면 자세히 알려주세요. 

해당 정보를 바탕으로 더 잘 도와드릴 수 있을 테니까요!
AI: 당신이 '1234'이라고 말씀하신 것을 기억하고 있습니다. 제가 이전 대화 내용을 통해 이를 파악했기 때문이에요. 혹시 추가로 궁금한 점이 있으시면 언제든지 말해주세요!
AI: 안녕하세요! 저는 당신의 메시지를 하나씩 실시간으로 확인하고 그에 맞춰 답변하는 방식이라, 대화의 흐름을 자연스럽게 이어갈 수 있습니다. 하지만 이전 세션에 대한 장기 기억은 없으며, 매번 새로운 질문이 들어오면 그때부터 다시 시작됩니다. 궁금한 점이 있으면 언제든지 물어보세요!
AI: 저는 대화의 맥락을 이해하기 위해 이전 메시지를 분석하며 답변을 준비합니다.  
1234 라는 숫자나 간단한 입력은 특정 정보를 저장할 수 있지만, 저는 실시간으로 질문하신 내용을 참고하여 최선을 다해 도와드리겠습니다.  
어떤 도움이 필요하신지 알려주시면 바로 응답해 드리겠습니다!
AI: 사내 도우미로서 친절하게 답변드립니다. 저는 각 대화마다 새로운 세션으로 시작되므로 이전 채팅의 내용을 직접 기억하지는 못합니다. 다만, 현재 대화창 내에서 방금 말씀하신 내용을 바탕으로 자연스럽게 응답하고 있습니다. 더 궁금한 점이 있거나 도와드릴 일이 있다면 언제든 말해 주세요!
